# Task 2: Citation Mapping - BERT Training

**Model:** bert-base-uncased (Sequence Classification)

**Bài toán:** Cho text có [CITATION_X] + danh sách candidate papers (title + abstract) → Predict [CITATION_X] map với paper nào?

**Approach:** Với mỗi cặp (context, candidate paper) → model predict match (1) hoặc không match (0)

**Dataset:** task2-citation-mapping

---

## 1. Setup & Imports

In [1]:
import transformers, datasets, accelerate
print(f"✅ transformers: {transformers.__version__}")
print(f"✅ datasets: {datasets.__version__}")
print(f"✅ accelerate: {accelerate.__version__}")

✅ transformers: 5.0.0
✅ datasets: 4.8.3
✅ accelerate: 1.12.0


## 2. Wandb Login

In [2]:
import wandb
import os

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    key = secrets.get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = key
    wandb.login(key=key, relogin=True)
    print("✅ Wandb logged in")
except Exception as e:
    print(f"⚠️ Wandb login failed: {e}")
    os.environ["WANDB_MODE"] = "disabled"
    print("⚠️ Wandb disabled — training will continue without logging")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tathiyennhi (tathiyennhi-hcmus) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Wandb logged in


## 3. Data Paths

In [3]:
import os

DATA_ROOT = "/kaggle/input/datasets/tathiyennhi/task2-citation-mapping/task2"

train_path = os.path.join(DATA_ROOT, "train")
val_path = os.path.join(DATA_ROOT, "val")

# Uncomment nếu cần xem cấu trúc thư mục
# for root, dirs, files in os.walk("/kaggle/input"):
#     print(root, dirs[:5], files[:5])

train_count = len([f for f in os.listdir(train_path) if f.endswith('.label')])
val_count = len([f for f in os.listdir(val_path) if f.endswith('.label')])

print(f"✅ Train: {train_count:,} files")
print(f"✅ Val: {val_count:,} files")

✅ Train: 55,556 files
✅ Val: 3,000 files


In [4]:
import json
from pathlib import Path

sample_label = sorted(Path(train_path).glob("*.label"))[0]
sample_in = sample_label.with_suffix(".in")

print("=== FILE .in ===")
with open(sample_in) as f:
    in_data = json.load(f)
print(f"Keys: {list(in_data.keys())}")
print(f"Text (first 200): {in_data.get('text', '')[:200]}")
print(f"Candidates: {len(in_data.get('citation_candidates', []))}")
print(f"Bib entries: {len(in_data.get('bib_entries', {}))}")

print("\n=== FILE .label ===")
with open(sample_label) as f:
    label_data = json.load(f)
print(f"Keys: {list(label_data.keys())}")
print(f"correct_citation: {label_data.get('correct_citation', {})}")

=== FILE .in ===
Keys: ['text', 'citation_candidates', 'bib_entries']
Text (first 200): We highlight our contributions by comparing with [CITATION_1] . In [CITATION_2] the state-wise ambiguity set is restricted to the following form:C s = {μ s |μ s (O i s ) ≥ α i s ∀ i = 1, . . . , n s }
Candidates: 10
Bib entries: 10

=== FILE .label ===
Keys: ['text', 'correct_citation', 'bib_entries']
correct_citation: {'[CITATION_1]': '7229756', '[CITATION_2]': '7229756', '[CITATION_3]': '16625241'}


## 4. Load Data → Training Format

**Strategy:** Với mỗi citation trong document:
- Lấy context theo CONTEXT_MODE: 'full' (cả đoạn) hoặc 'window_N' (N câu xung quanh marker)
- Với mỗi candidate paper: tạo 1 training example
  - input = [CLS] context [SEP] candidate title. candidate abstract [SEP]
  - label = 1 nếu candidate đúng, 0 nếu sai

**Ablation:** Đổi CONTEXT_MODE để chạy experiments khác nhau

In [5]:
import json
import re
import random
from pathlib import Path
from datasets import Dataset

# ═══════════════════════════════════════════════════════════
# ⚙️ CONTEXT MODE — thay đổi giá trị này để chạy ablation
# ═══════════════════════════════════════════════════════════
# 'full'     → dùng toàn bộ đoạn text (baseline, có nhiễu)
# 'window_0' → chỉ câu chứa citation marker
# 'window_1' → câu chứa + 1 câu trước/sau
# 'window_2' → câu chứa + 2 câu trước/sau
CONTEXT_MODE = 'full'
# ═══════════════════════════════════════════════════════════


def get_context(text, citation_id, mode='full'):
    if mode == 'full':
        return text
    
    window = int(mode.split('_')[1])
    sentences = re.split(r'(?<=[.!?])\s+', text)
    
    target_idx = -1
    for i, sent in enumerate(sentences):
        if citation_id in sent:
            target_idx = i
            break
    
    if target_idx == -1:
        return text
    
    start = max(0, target_idx - window)
    end = min(len(sentences), target_idx + window + 1)
    return ' '.join(sentences[start:end])


def load_task2_data(data_dir, max_files=None, neg_ratio=3, context_mode='full'):
    data_path = Path(data_dir)
    label_files = sorted(data_path.glob('*.label'))
    
    if max_files:
        label_files = label_files[:max_files]
    
    total_files = len(label_files)
    print(f'📊 Loading {total_files:,} files | Mode: {context_mode}')
    
    examples = []
    skipped = 0
    stats = {'positive': 0, 'negative': 0}
    
    for file_idx, label_file in enumerate(label_files):
        if (file_idx + 1) % 1000 == 0:
            print(f'⏳ {file_idx+1:,}/{total_files:,} | Examples: {len(examples):,}')
        
        in_file = label_file.with_suffix('.in')
        
        try:
            with open(in_file) as f:
                in_data = json.load(f)
            with open(label_file) as f:
                label_data = json.load(f)
        except:
            skipped += 1
            continue
        
        text = in_data.get('text', '')
        if not text:
            skipped += 1
            continue
        
        candidates = in_data.get('citation_candidates', [])
        bib_entries = in_data.get('bib_entries', {})
        correct_citation = label_data.get('correct_citation', {})
        
        if not correct_citation or not candidates or not bib_entries:
            skipped += 1
            continue
        
        for citation_id, correct_paper_id in correct_citation.items():
            context = get_context(text, citation_id, mode=context_mode)
            
            if correct_paper_id in bib_entries:
                paper = bib_entries[correct_paper_id]
                paper_text = f"{paper.get('title', '')}. {paper.get('abstract', '')}"
                examples.append({'text_a': context, 'text_b': paper_text, 'label': 1})
                stats['positive'] += 1
            
            neg_candidates = [c for c in candidates if c != correct_paper_id and c in bib_entries]
            neg_sample = random.sample(neg_candidates, min(neg_ratio, len(neg_candidates)))
            
            for neg_paper_id in neg_sample:
                paper = bib_entries[neg_paper_id]
                paper_text = f"{paper.get('title', '')}. {paper.get('abstract', '')}"
                examples.append({'text_a': context, 'text_b': paper_text, 'label': 0})
                stats['negative'] += 1
    
    print(f'\n✅ {len(examples):,} examples | Pos: {stats["positive"]:,} | Neg: {stats["negative"]:,} | Skip: {skipped}')
    return examples


print('=' * 60)
print(f'CONTEXT MODE: {CONTEXT_MODE}')
print('=' * 60)

train_examples = load_task2_data(train_path, neg_ratio=3, context_mode=CONTEXT_MODE)
val_examples = load_task2_data(val_path, neg_ratio=3, context_mode=CONTEXT_MODE)

train_dataset = Dataset.from_list(train_examples)
val_dataset = Dataset.from_list(val_examples)

print(f'\n✅ Train: {len(train_dataset):,} | Val: {len(val_dataset):,}')

CONTEXT MODE: full
📊 Loading 55,556 files | Mode: full
⏳ 1,000/55,556 | Examples: 9,354
⏳ 2,000/55,556 | Examples: 19,564
⏳ 3,000/55,556 | Examples: 28,466
⏳ 4,000/55,556 | Examples: 37,282
⏳ 5,000/55,556 | Examples: 46,464
⏳ 6,000/55,556 | Examples: 55,701
⏳ 7,000/55,556 | Examples: 64,876
⏳ 8,000/55,556 | Examples: 74,497
⏳ 9,000/55,556 | Examples: 83,561
⏳ 10,000/55,556 | Examples: 92,538
⏳ 11,000/55,556 | Examples: 101,509
⏳ 12,000/55,556 | Examples: 110,616
⏳ 13,000/55,556 | Examples: 120,131
⏳ 14,000/55,556 | Examples: 129,434
⏳ 15,000/55,556 | Examples: 137,777
⏳ 16,000/55,556 | Examples: 146,906
⏳ 17,000/55,556 | Examples: 156,140
⏳ 18,000/55,556 | Examples: 164,600
⏳ 19,000/55,556 | Examples: 173,365
⏳ 20,000/55,556 | Examples: 182,450
⏳ 21,000/55,556 | Examples: 191,771
⏳ 22,000/55,556 | Examples: 201,003
⏳ 23,000/55,556 | Examples: 210,029
⏳ 24,000/55,556 | Examples: 218,620
⏳ 25,000/55,556 | Examples: 227,619
⏳ 26,000/55,556 | Examples: 236,468
⏳ 27,000/55,556 | Examples: 2

## 5. Tokenization

In [6]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Tokenizer: {MODEL_NAME}")


def tokenize_function(examples):
    return tokenizer(
        examples["text_a"],
        examples["text_b"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
    )


print("Tokenizing train...")
train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=["text_a", "text_b"])

print("Tokenizing val...")
val_tokenized = val_dataset.map(tokenize_function, batched=True, remove_columns=["text_a", "text_b"])

print(f"\n✅ Train: {len(train_tokenized):,} | Val: {len(val_tokenized):,}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer: bert-base-uncased
Tokenizing train...


Map:   0%|          | 0/511596 [00:00<?, ? examples/s]

Tokenizing val...


Map:   0%|          | 0/27982 [00:00<?, ? examples/s]


✅ Train: 511,596 | Val: 27,982


## 6. Load Model

In [7]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
print(f"✅ Model: {MODEL_NAME} (num_labels=2)")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model: bert-base-uncased (num_labels=2)


## 7. Metrics

In [8]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", pos_label=1)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}


print("✅ Metrics defined")

✅ Metrics defined


## 8. Training Configuration

In [9]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import wandb

WANDB_PROJECT = "task2-citation-mapping"
CHECKPOINT_DIR = "/kaggle/working/checkpoints/task2_bert"
SAVE_DIR = "/kaggle/working/task2_bert_final"

try:
    if wandb.run is None:
        wandb.init(
            project=WANDB_PROJECT,
            name=f"bert-base-lr3e5-{CONTEXT_MODE}",
            config={"model": MODEL_NAME, "learning_rate": 3e-5, "max_steps": 5000,
                    "warmup_steps": 500, "context_mode": CONTEXT_MODE, "neg_ratio": 3},
            resume="allow",
        )
    report_to = "wandb"
    print("✅ Wandb initialized")
except:
    report_to = "none"
    print("⚠️ Wandb not available")


training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    max_steps=5000,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_dir="/kaggle/working/logs",
    logging_steps=100,
    report_to=report_to,
    fp16=True,
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"\n✅ Trainer ready | max_steps={training_args.max_steps} | LR={training_args.learning_rate} | batch=32")

wandb: setting up run x9vgnb5m
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260404_181632-x9vgnb5m
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run bert-base-lr3e5-full
wandb: ⭐️ View project at https://wandb.ai/tathiyennhi-hcmus/task2-citation-mapping
wandb: 🚀 View run at https://wandb.ai/tathiyennhi-hcmus/task2-citation-mapping/runs/x9vgnb5m


✅ Wandb initialized


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



✅ Trainer ready | max_steps=5000 | LR=3e-05 | batch=32


## 9. Train

In [10]:
print("=" * 60)
print("🚀 TRAINING BERT FOR CITATION MAPPING")
print("=" * 60)

trainer.train()

print("\n✅ Training complete!")

🚀 TRAINING BERT FOR CITATION MAPPING


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
500,2.041536,1.073225,0.705561,0.439264,0.586037,0.502145
1000,1.945097,0.995181,0.767708,0.622814,0.211001,0.315213
1500,1.957570,0.972090,0.765528,0.551369,0.400423,0.463927
2000,1.949276,0.958845,0.770209,0.687075,0.170945,0.273775


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


✅ Training complete!


## 10. Evaluation

In [11]:
print("📊 VALIDATION RESULTS")
print("=" * 60)

eval_results = trainer.evaluate()

for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

print("=" * 60)
print(f"\n✅ Accuracy:  {eval_results.get('eval_accuracy', 0):.2%}")
print(f"✅ Precision: {eval_results.get('eval_precision', 0):.2%}")
print(f"✅ Recall:    {eval_results.get('eval_recall', 0):.2%}")
print(f"✅ F1:        {eval_results.get('eval_f1', 0):.2%}")

📊 VALIDATION RESULTS


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


eval_loss: 1.0712
eval_accuracy: 0.7067
eval_precision: 0.4405
eval_recall: 0.5839
eval_f1: 0.5022
eval_runtime: 493.6317
eval_samples_per_second: 56.6860
eval_steps_per_second: 0.8870
epoch: 0.2502

✅ Accuracy:  70.67%
✅ Precision: 44.05%
✅ Recall:    58.39%
✅ F1:        50.22%


## 11. Save Model & Report

In [12]:
import os, shutil

os.makedirs(SAVE_DIR, exist_ok=True)
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
shutil.make_archive(SAVE_DIR, 'zip', SAVE_DIR)
print(f"✅ Model saved: {SAVE_DIR}")

summary = f"""============================================================
EXPERIMENT RESULTS - Task 2 Citation Mapping
============================================================
Model:        {MODEL_NAME}
Context:      {CONTEXT_MODE}
Max length:   {MAX_LENGTH}
Max steps:    {training_args.max_steps}
LR:           {training_args.learning_rate}
Batch:        {training_args.per_device_train_batch_size}x{training_args.gradient_accumulation_steps}={training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}
Warmup:       {training_args.warmup_steps}
Weight decay: {training_args.weight_decay}
Train data:   {len(train_tokenized):,}
Val data:     {len(val_tokenized):,}
------------------------------------------------------------
Accuracy:     {eval_results.get('eval_accuracy', 0):.4f}
Precision:    {eval_results.get('eval_precision', 0):.4f}
Recall:       {eval_results.get('eval_recall', 0):.4f}
F1:           {eval_results.get('eval_f1', 0):.4f}
Eval loss:    {eval_results.get('eval_loss', 0):.4f}
============================================================"""

print(summary)

os.makedirs("/kaggle/working/results", exist_ok=True)
with open("/kaggle/working/results/task2_report.txt", "w") as f:
    f.write(summary)

try:
    wandb.finish()
except:
    pass

print("\n🎉 TASK 2 TRAINING COMPLETE!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

wandb: updating run metadata


✅ Model saved: /kaggle/working/task2_bert_final
EXPERIMENT RESULTS - Task 2 Citation Mapping
Model:        bert-base-uncased
Context:      full
Max length:   512
Max steps:    5000
LR:           3e-05
Batch:        16x2=32
Warmup:       500
Weight decay: 0.01
Train data:   511,596
Val data:     27,982
------------------------------------------------------------
Accuracy:     0.7067
Precision:    0.4405
Recall:       0.5839
F1:           0.5022
Eval loss:    1.0712


wandb: uploading summary, console lines 38-58
wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁█▇█▁
wandb:                 eval/f1 █▂▇▁█
wandb:               eval/loss █▃▂▁█
wandb:          eval/precision ▁▆▄█▁
wandb:             eval/recall █▂▅▁█
wandb:            eval/runtime ▃▆█▄▁
wandb: eval/samples_per_second ▆▃▁▅█
wandb:   eval/steps_per_second ▆▃▁▆█
wandb:             train/epoch ▁▁▂▂▂▂▃▃▄▄▄▄▅▅▅▆▆▆▇▇▇█████
wandb:       train/global_step ▁▁▂▂▂▂▃▃▄▄▄▄▅▅▅▆▆▆▇▇▇█████
wandb:                      +3 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.70667
wandb:                 eval/f1 0.50218
wandb:               eval/loss 1.07123
wandb:          eval/precision 0.44052
wandb:             eval/recall 0.58392
wandb:            eval/runtime 493.6317
wandb: eval/samples_per_second 56.686
wandb:   eval/steps_per_second 0.887
wandb:              total_flos 3.367821508608e+16
wandb:             train/epoch 0.25019
wandb:                      +8 ...
wandb: 
wandb: 🚀 


🎉 TASK 2 TRAINING COMPLETE!
